In [ ]:
from pathlib import Path

import numpy as np
import plotly.express as px
import polars as pl
import torch
from numpy.typing import NDArray
from sklearn.multioutput import MultiOutputClassifier
from tqdm import tqdm
from torch.utils.data import DataLoader
from xgboost import XGBClassifier

from tarp.cli.logging import Console
from tarp.model.backbone.pretrained.esm1b import FrozenEsm1bEncoder
from tarp.services.datasets.classification.multilabel import (
    MultiLabelClassificationDataset,
)
from tarp.services.datasources.sequence import TabularSequenceSource
from tarp.services.evaluation.classification.multilabel import MultiLabelMetrics
from tarp.services.preprocessing.augmentation import (
    CompositeAugmentation,
)
from tarp.services.preprocessing.augmentation.protein import (
    InsertionDeletion,
    RandomMutation,
)
from tarp.services.tokenizers.pretrained.esm1b import Esm1bTokenizer

In [ ]:
classification_head = MultiOutputClassifier(
    XGBClassifier(
        learning_rate=0.1,
        max_depth=7,
        n_estimators=200,
        objective="binary:logistic",
        tree_method= 'gpu_hist' if torch.cuda.is_available() else 'hist',
    )
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pretrained_encoder = FrozenEsm1bEncoder().to(device)
tokenizer = Esm1bTokenizer()
label_columns = pl.read_csv(Path("../temp/data/cache/labels.csv")).to_series().to_list()
label_columns.remove("non_amr")

Console.info("Model and classification head initialized")

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm1b_t33_650M_UR50S and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[INFO]	2025-12-12 15:38:00,075 - Model and classification head initialized


In [5]:
multilabel_classification_train = MultiLabelClassificationDataset(
    (
        TabularSequenceSource(
            source=Path("../temp/data/processed/card_amr.train.parquet"),
        )
        # + FastaSliceSource(
        #     directory=Path("temp/data/external/sequences/proteins"),
        #     metadata=Path("temp/data/processed/fine_tuning.train.parquet"),
        #     key_column="protein_accession.version",
        #     start_column="?",
        #     end_column="?",
        #     sequence_column="protein_sequence",
        # )
    ),
    tokenizer=tokenizer,
    sequence_column="protein_sequence",
    label_columns=label_columns,
    maximum_sequence_length=200,
    augmentation=CompositeAugmentation(
        [
            RandomMutation(),
            InsertionDeletion(),
        ]
    ),
)

multilabel_classification_valid = MultiLabelClassificationDataset(
    (
        TabularSequenceSource(
            source=Path("../temp/data/processed/card_amr.valid.parquet"),
        )
        # + FastaSliceSource(
        #     directory=Path("temp/data/external/sequences/proteins"),
        #     metadata=Path("temp/data/processed/fine_tuning.valid.parquet"),
        #     key_column="protein_accession.version",
        #     start_column="?",
        #     end_column="?",
        #     sequence_column="protein_sequence",
        # )
    ),
    tokenizer=tokenizer,
    sequence_column="protein_sequence",
    label_columns=label_columns,
    maximum_sequence_length=200,
)
Console.info("Datasets initialized")

[INFO]	2025-12-12 15:38:00,106 - Datasets initialized


In [6]:
train_loader = DataLoader(
    multilabel_classification_train,
    batch_size=16,
    shuffle=True,
    num_workers=4,
)

valid_loader = DataLoader(
    multilabel_classification_valid,
    batch_size=16,
    shuffle=False,
    num_workers=4,
)

In [7]:

x_embeddings: list[torch.Tensor] = []
y_labels: list[torch.Tensor] = []

# Encode training set
for batch in tqdm(train_loader, desc="Encoding training"):
    input_ids = batch["sequence"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"]

    with torch.no_grad():
        pooled = pretrained_encoder.encode(
            input_ids,
            attention_mask=attention_mask,
            return_sequence=False,
        )  # (B, hidden)

    # Keep as torch tensors
    x_embeddings.append(pooled.cpu())
    y_labels.append(labels.cpu())

Encoding training: 100%|██████████| 631/631 [07:58<00:00,  1.32it/s]


In [31]:
stacked_x = torch.cat(x_embeddings, dim=0).numpy()  # shape: (num_samples, hidden)
stacked_y = torch.cat(y_labels, dim=0).numpy() # shape: (num_samples, num_labels)

In [9]:
Console.info("Starting training classification head")
classification_head.fit(stacked_x, stacked_y)
Console.info("Classification head training complete")

[INFO]	2025-12-12 15:45:58,465 - Starting training classification head
[INFO]	2025-12-12 15:53:02,655 - Classification head training complete


In [ ]:
# Pickle the training and validation datasets
import pickle
with open("../temp/multilabel_classification_train.pkl", "wb") as f:
    pickle.dump(stacked_x, f)
with open("../temp/multilabel_classification_valid.pkl", "wb") as f:
    pickle.dump(stacked_y, f)
Console.info("Pickled training dataset")

In [10]:
x_val_embeddings: list[NDArray] = []
y_val_labels: list[NDArray] = []

for batch in tqdm(valid_loader, desc="Encoding validation"):
    input_ids = batch["sequence"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"]

    with torch.no_grad():
        pooled = pretrained_encoder.encode(
            input_ids,
            attention_mask=attention_mask,
            return_sequence=False,
        )  # (B, hidden)

    x_val_embeddings.append(pooled.cpu().numpy())
    y_val_labels.append(labels.numpy())

Encoding validation: 100%|██████████| 78/78 [01:09<00:00,  1.12it/s]


In [11]:
# np.vstack to create 2D arrays
stacked_x_val = np.concatenate(x_val_embeddings, axis=0)
stacked_y_val = np.concatenate(y_val_labels, axis=0)

In [12]:
Console.info("Starting validation")
val_predictions = classification_head.predict_proba(stacked_x_val)
val_predictions_stacked = np.column_stack([p[:, 1] for p in val_predictions])

thresholds = np.linspace(0.0, 1.0, 101)
rows = []

for thr in thresholds:
    metrics = MultiLabelMetrics(threshold=thr).compute(
        torch.as_tensor(val_predictions_stacked),
        torch.as_tensor(stacked_y_val),
    )
    rows.append({"threshold": thr, **{k: float(v) for k, v in metrics.items()}})

[INFO]	2025-12-12 15:54:12,544 - Starting validation
[WARN]	2025-12-12 15:54:12,877 - Skipping 15 invalid classes (all-zeros or all-ones) for ROC AUC.
[WARN]	2025-12-12 15:54:13,169 - Skipping 15 invalid classes (all-zeros or all-ones) for ROC AUC.
[WARN]	2025-12-12 15:54:13,445 - Skipping 15 invalid classes (all-zeros or all-ones) for ROC AUC.
[WARN]	2025-12-12 15:54:13,735 - Skipping 15 invalid classes (all-zeros or all-ones) for ROC AUC.
[WARN]	2025-12-12 15:54:13,994 - Skipping 15 invalid classes (all-zeros or all-ones) for ROC AUC.
[WARN]	2025-12-12 15:54:14,261 - Skipping 15 invalid classes (all-zeros or all-ones) for ROC AUC.
[WARN]	2025-12-12 15:54:14,516 - Skipping 15 invalid classes (all-zeros or all-ones) for ROC AUC.
[WARN]	2025-12-12 15:54:14,682 - Skipping 15 invalid classes (all-zeros or all-ones) for ROC AUC.
[WARN]	2025-12-12 15:54:14,856 - Skipping 15 invalid classes (all-zeros or all-ones) for ROC AUC.
[WARN]	2025-12-12 15:54:15,017 - Skipping 15 invalid classes (all

In [15]:
df = pl.DataFrame(rows)

metric_cols = [c for c in df.columns if c != "threshold"]

df_long = df.unpivot(
    on=metric_cols, index="threshold", variable_name="metric", value_name="value"
)

# Plotly expects pandas df
fig = px.line(
    df_long.to_pandas(),
    x="threshold",
    y="value",
    color="metric",
    title="Multi-Label Metrics Across Thresholds",
    markers=True,
)

target_metric = "f1"

best_threshold = (
    df.sort(target_metric, descending=True)
      .select("threshold")
      .head(1)
      .item()
)

best_metric_value = (
    df.sort(target_metric, descending=True)
      .select(target_metric)
      .head(1)
      .item()
)

Console.info(
    f"Best threshold: {best_threshold:.2f} with {target_metric}: {best_metric_value:.4f}"
)
fig.show()

[INFO]	2025-12-12 15:55:31,818 - Best threshold: 0.52 with f1: 0.5365


In [16]:
from sklearn.metrics import multilabel_confusion_matrix

threshold = best_threshold  # or set manually
y_true = stacked_y_val
y_pred = (val_predictions_stacked >= threshold).astype(int)

mcm = multilabel_confusion_matrix(y_true, y_pred)

In [26]:
true = stacked_y_val
pred = (val_predictions_stacked >= best_threshold).astype(int)

n_classes = true.shape[1]

conf = np.zeros((n_classes, n_classes), dtype=int)

for a in range(n_classes):
    for b in range(n_classes):
        conf[a, b] = np.sum((true[:, a] == 1) & (pred[:, b] == 1))

In [28]:
import plotly.express as px

px.imshow(
    conf,
    x=label_columns,
    y=label_columns,
    labels=dict(x="Predicted Class", y="True Class"),
    title="Class–Class Confusion / Bias Matrix (Row Normalized)",
).update_layout(
    autosize=False,
    width=800,
    height=800,
)

In [33]:
# What about on training set?
true = stacked_y
pred = (np.column_stack([p[:, 1] for p in classification_head.predict_proba(stacked_x)]) > best_threshold).astype(int)

conf = np.zeros((n_classes, n_classes), dtype=int)

for a in range(n_classes):
    for b in range(n_classes):
        conf[a, b] = np.sum((true[:, a] == 1) & (pred[:, b] == 1))


px.imshow(
    conf,
    x=label_columns,
    y=label_columns,
    labels=dict(x="Predicted Class", y="True Class"),
    title="Class–Class Confusion / Bias Matrix (Row Normalized)",
).update_layout(
    autosize=False,
    width=800,
    height=800,
)

In [ ]:
# Use the best threshold to compute final metrics on validation set
final_metrics = MultiLabelMetrics(threshold=best_threshold).compute(
    torch.as_tensor(val_predictions_stacked),
    torch.as_tensor(stacked_y_val),
)